In [6]:
"""
semantic_cache_demo.py
Wire up your separated modules into a full demo:
  session/context -> embedding (text-embedding-004) -> FAISS -> cache hit/miss.
Includes auto-picking a working text model for your API key.

Run:
  python semantic_cache_demo.py --threshold 0.86 --k 4 --scenario agri
"""

from __future__ import annotations
import os, time, argparse

from session_store import SessionStore
from context_builder import build_context_text
from embedder import Embedder
from cache_index import CacheIndex

import google.generativeai as genai
from dotenv import load_dotenv, find_dotenv

# Configure API key
load_dotenv(find_dotenv(usecwd=True), override=True)
API_KEY = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")
if not API_KEY:
    raise RuntimeError("Missing GOOGLE_API_KEY / GEMINI_API_KEY in .env")
genai.configure(api_key=API_KEY)

EMBED_MODEL = "text-embedding-004"   # used inside your Embedder class

# 1) Auto-pick a working text model (prefers 2.5 series)
def _normalize(name: str) -> str:
    return name if name.startswith("models/") else f"models/{name}"

def pick_working_text_model(preferred=None, require_cap="generateContent"):
    pinned = os.getenv("GEN_MODEL_NAME")
    if pinned:
        try:
            m = genai.GenerativeModel(pinned); m.generate_content("ok.")
            return pinned, m
        except Exception:
            pass

    default_preference = [
        "models/gemini-2.5-flash",
        "models/gemini-2.5-pro",
        "models/gemini-2.0-flash",
        "models/gemini-2.0-pro",
        "models/gemini-1.0-pro",
    ]
    pref = [_normalize(n) for n in (preferred or default_preference)]

    names_with_cap = []
    try:
        for m in genai.list_models():
            caps = set(getattr(m, "supported_generation_methods", []) or [])
            if require_cap in caps:
                names_with_cap.append(m.name)
    except Exception:
        names_with_cap = pref[:]

    ordered = [n for n in pref if n in names_with_cap] + \
              [n for n in names_with_cap if n not in pref]

    last_err = None
    for name in ordered:
        try:
            model = genai.GenerativeModel(name)
            model.generate_content("ok.")
            return name, model
        except Exception as e:
            last_err = e

    try:
        name = "models/gemini-1.0-pro"
        model = genai.GenerativeModel(name); model.generate_content("ok.")
        return name, model
    except Exception as e:
        raise RuntimeError(f"No working text model found. Last error: {e}")

GEN_MODEL_NAME, TEXT_MODEL = pick_working_text_model()
print(f"[Model] Using text model: {GEN_MODEL_NAME}")

def llm_answer(context_text: str) -> tuple[str, float]:
    """Call the chosen text model and return (answer, latency_s)."""
    t0 = time.time()
    resp = TEXT_MODEL.generate_content(context_text)
    return resp.text, time.time() - t0

SCENARIOS = {
    "agri": [
        "What is the impact of climate change on corn yields in the US?",
        "How does global warming affect the productivity of maize crops?",
        "What about wheat?",
    ],
    "finance": [
        "Explain compound interest for a 10-year horizon at 5%.",
        "Could you show the same for 7%?",
        "What if I invest monthly 200 dollars?",
    ],
    "retail": [
        "Give me 3 ways to reduce retail shrink.",
        "How to cut shoplifting losses in supermarkets?",
        "Any low-cost prevention ideas?",
    ],
}

def run_demo(queries: list[str], k_turns: int, threshold: float):
    embedder = Embedder()   # your class already uses EMBED_MODEL internally
    dim = embedder.dim()
    index = CacheIndex(dim)
    sessions = SessionStore()

    hits = misses = 0
    lat_hit, lat_miss = [], []
    session_id = "demo"

    for q in queries:
        hist = sessions.history(session_id, k=k_turns)
        ctx  = build_context_text(hist, q)
        vec  = embedder.embed(ctx)
        score, payload = index.search(vec, topk=1)

        if payload and score >= threshold:
            ans = payload["answer"]
            hits += 1; lat_hit.append(0.01)
            print(f"[HIT   score={score:.3f}] {q}")
        else:
            ans, latency = llm_answer(ctx)
            misses += 1; lat_miss.append(latency)
            index.add(vec, {"session_id": session_id, "context": ctx, "answer": ans})
            print(f"[MISS  score={score:.3f}  {latency:.2f}s] {q}")

        sessions.add_turn(session_id, "user", q)
        sessions.add_turn(session_id, "assistant", ans)

    total = hits + misses
    print("\n=== Metrics ===")
    print(f"Using model: {GEN_MODEL_NAME}")
    print(f"Requests: {total}")
    print(f"Hit rate: {hits/total:.2f}")
    if lat_hit:  print(f"Avg latency (hit):  {sum(lat_hit)/len(lat_hit):.3f}s")
    if lat_miss: print(f"Avg latency (miss): {sum(lat_miss)/len(lat_miss):.3f}s")

def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--threshold", type=float, default=float(os.getenv("THRESHOLD", 0.86)))
    p.add_argument("--k",         type=int,   default=int(os.getenv("K_TURNS", 4)))
    p.add_argument("--scenario",  type=str,   default="agri",
                   choices=list(SCENARIOS.keys()) + ["all"])
    return p.parse_args()

if __name__ == "__main__":
    args = parse_args()
    if args.scenario == "all":
        for name, qs in SCENARIOS.items():
            print(f"\n=== Scenario: {name} ===")
            run_demo(qs, k_turns=args.k, threshold=args.threshold)
    else:
        run_demo(SCENARIOS[args.scenario], k_turns=args.k, threshold=args.threshold)


[Model] Using text model: models/gemini-2.5-flash


usage: ipykernel_launcher.py [-h] [--threshold THRESHOLD] [--k K]
                             [--scenario {agri,finance,retail,all}]
ipykernel_launcher.py: error: unrecognized arguments: --f=c:\Users\USER\AppData\Roaming\jupyter\runtime\kernel-v32bf0467a3b54b838f76f8812e896e1df4d420147.json


SystemExit: 2

d:\Yv_material\UIUC\DSRS\online_accessment\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
